In [2]:
with open("The-Universe-in-Your-Hand.txt", encoding="utf-8") as f:
    text = f.read()

In [3]:
print("length of book is: ", len(text))

length of book is:  619804


In [4]:
print(text[:1000])

Foreword





Before we start, there are two things I would like to share with you.

The first is a promise, the second is an ambition.

The promise is that the book contains only one equation.

Here it is:



E=mc2





The ambition, my ambition, is that in this book I will not leave any readers behind.

You are about to start a journey through the universe as it is understood by science today. It is my deepest belief that we can all understand this stuff.

And that journey begins a very long way from home, on the other side of the world.





Part One


The Cosmos





1 | A Silent Boom





Picture yourself on a faraway volcanic island on a warm, cloudless summer night. The surrounding ocean is as still as a lake. Only the tiniest of waves wash against the white sand. All is quiet. You are lying on the beach. Your eyes are closed. The warm, sun-baked sand heats up air saturated with sweet, exotic scents. There is peace all around.

A wild shriek in the distance makes you jump and st

In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print("vocab size is: ", vocab_size)


 !&()*,-./0123456789:;=?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz|£©­°Öçíîö–—‘’“”•
vocab size is:  94


In [ ]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] #string to integers
decode = lambda l: ''.join([itos[i] for i in l]) #integers to string

# could also use tiktoken (what chatgpt uses)

In [7]:
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([619804]) torch.int64
tensor([30, 65, 68, 55, 73, 65, 68, 54,  0,  0,  0,  0,  0,  0, 26, 55, 56, 65,
        68, 55,  1, 73, 55,  1, 69, 70, 51, 68, 70,  7,  1, 70, 58, 55, 68, 55,
         1, 51, 68, 55,  1, 70, 73, 65,  1, 70, 58, 59, 64, 57, 69,  1, 33,  1,
        73, 65, 71, 62, 54,  1, 62, 59, 61, 55,  1, 70, 65,  1, 69, 58, 51, 68,
        55,  1, 73, 59, 70, 58,  1, 75, 65, 71,  9,  0,  0, 44, 58, 55,  1, 56,
        59, 68, 69, 70,  1, 59, 69,  1, 51,  1, 66, 68, 65, 63, 59, 69, 55,  7,
         1, 70, 58, 55,  1, 69, 55, 53, 65, 64, 54,  1, 59, 69,  1, 51, 64,  1,
        51, 63, 52, 59, 70, 59, 65, 64,  9,  0,  0, 44, 58, 55,  1, 66, 68, 65,
        63, 59, 69, 55,  1, 59, 69,  1, 70, 58, 51, 70,  1, 70, 58, 55,  1, 52,
        65, 65, 61,  1, 53, 65, 64, 70, 51, 59, 64, 69,  1, 65, 64, 62, 75,  1,
        65, 64, 55,  1, 55, 67, 71, 51, 70, 59, 65, 64,  9,  0,  0, 32, 55, 68,
        55,  1, 59, 70,  1, 59, 69, 21,  0,  0,  0,  0, 29, 23, 63, 53, 13,  0,
       

In [8]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [9]:
block_size = 8
train_data[:block_size+1]

tensor([30, 65, 68, 55, 73, 65, 68, 54,  0])

In [ ]:
block_size = 8
train_data[:block_size+1]
# will make a prediction at each of these positions for the following character

tensor([30, 65, 68, 55, 73, 65, 68, 54,  0])

In [11]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([30]) the target: 65
when input is tensor([30, 65]) the target: 68
when input is tensor([30, 65, 68]) the target: 55
when input is tensor([30, 65, 68, 55]) the target: 73
when input is tensor([30, 65, 68, 55, 73]) the target: 65
when input is tensor([30, 65, 68, 55, 73, 65]) the target: 68
when input is tensor([30, 65, 68, 55, 73, 65, 68]) the target: 54
when input is tensor([30, 65, 68, 55, 73, 65, 68, 54]) the target: 0


In [12]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[ 1, 53, 51, 64,  1, 51, 62, 69],
        [58, 55,  1, 68, 65, 53, 61,  1],
        [56, 51, 69, 70,  1, 51, 68, 65],
        [62, 62, 59, 64, 57,  1, 54, 65]])
targets:
torch.Size([4, 8])
tensor([[53, 51, 64,  1, 51, 62, 69, 65],
        [55,  1, 68, 65, 53, 61,  1, 51],
        [51, 69, 70,  1, 51, 68, 65, 71],
        [62, 59, 64, 57,  1, 54, 65, 73]])
----
when input is [1] the target: 53
when input is [1, 53] the target: 51
when input is [1, 53, 51] the target: 64
when input is [1, 53, 51, 64] the target: 1
when input is [1, 53, 51, 64, 1] the target: 51
when input is [1, 53, 51, 64, 1, 51] the target: 62
when input is [1, 53, 51, 64, 1, 51, 62] the target: 69
when input is [1, 53, 51, 64, 1, 51, 62, 69] the target: 65
when input is [58] the target: 55
when input is [58, 55] the target: 1
when input is [58, 55, 1] the target: 68
when input is [58, 55, 1, 68] the target: 65
when input is [58, 55, 1, 68, 65] the target: 53
when input is [58, 55, 1,

In [13]:
print(xb) # our input to the transformer

tensor([[ 1, 53, 51, 64,  1, 51, 62, 69],
        [58, 55,  1, 68, 65, 53, 61,  1],
        [56, 51, 69, 70,  1, 51, 68, 65],
        [62, 62, 59, 64, 57,  1, 54, 65]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

# 0 is the element for a new line character, so we will start with that 
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 94])
tensor(5.0486, grad_fn=<NllLossBackward0>)

)(Uqö‘!hsT•£mZ–A5 –b£d5a03öAkn-íÖí)(çl!u|TeCx|m?dî=fB•R1qT*Zg”6/!n-UHw2wG
XrtW5j©írlwcCÖaJ•*Z4de6vTX


loss was about log(-1/95) which is expected based on number of characters. it's not very good yet!

notice that the generated text is absolutely random :)

In [15]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [24]:
batch_size = 32
for steps in range(100000): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

5.06643533706665


In [25]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


cfJF3c*LA’°;VQ7uZ–­MI
EyW©&uQ9”O; Z5n
­YwjHN
X;Q3d(0’em•eYK9deP‘4/°Ö£)ILpsy:UHD1Ö9&°h2Kno6°3hl?dCÖçÖçzS.9GEmK;2K”­mW(12Dgqj“uíçz AIO:=WVM?Rt-xpA:suko—N*/u5u‘•wlp,2Q&wR£nLZg7“D:d1Jcaöx0­yVx(’3£JcöA=tqEd8”8of4|I|ne6.‘­yuö4I­Ub’2H-I2‘I6*&QI|DXT3V-î6ygnnDy—nnCI|ob)/RX°’4—ayZFK9C6v.NMNz3fjx’çq!W;!Z•*SSB;Fd,V
dbCÖPP’UqPSH(U’4I5|lDjxg”OQ­jMoi?JKD1Ug.1”y2î|p–k—J/a!yVC,,Nj;1g(nfqAQ£yEHKmiM““S—NW©|kM–ZîChkCUFaZÖC‘­gc
­HK“‘MIyPVKWL“­y:“Q7
g6R(p7!v0“pv—Rk:J1oC/7FQ5©M3mK)S*£hK4l•A’hta”cSN”2ÖXsN/!H/h-­*xu)wu.


the pieces are not talking to each other yet!